# NLLB Text Translation

Fresh Python 3.11+ CPU kernel. Downloads the exact official NLLB-200 distilled 600M checkpoint (approximately 2.5 GB), no API key. Research model, CC BY-NC 4.0; not production or certified translation. Preserve the exact original English text and five target variants. Fix unavailable local paths and use float32 on CPU instead of hardware-dependent bfloat16. Explicit 256-token generation budget avoids the truncated legacy output; all five pipeline translations are checked against direct generate and EOS. Translation fluency/dialect authenticity requires human review; structural checks do not certify linguistic quality. Original source code and outputs remain in appendices.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0", "transformers==4.57.6", "numpy==2.4.6", "Pillow==13.0.0", "sentencepiece==0.2.1", "certifi==2026.10.23", "ipywidgets==8.1.7"], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
import numpy as np
import torch
import transformers
from PIL import Image
torch.set_num_threads(2)
torch.manual_seed(42)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

## Example 0

In [ ]:
print("Pinned Transformers, PyTorch and SentencePiece dependencies installed in the active kernel.")

## Example 1

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
print("Translation pipeline and model classes imported.")

## Example 2

In [ ]:
MODEL_ID = "facebook/nllb-200-distilled-600M"
REVISION = "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=False, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, revision=REVISION, token=False, dtype=torch.float32).to("cpu").eval()
translator = pipeline(task="translation", model=model, tokenizer=tokenizer, device=-1)
print("Loaded:", MODEL_ID, "| revision:", REVISION)
print("Parameters:", sum(p.numel() for p in model.parameters()), "| dtype:", next(model.parameters()).dtype)

## Example 3

In [ ]:
text = """
Palestinians right to defend themselves is a fundamental aspect \
of their struggle for freedom and dignity. For decades, \
Palestinians have faced various forms of oppression, \
including occupation, displacement, and violence. \
In the face of such challenges, it is natural and justifiable for \
them to seek to protect themselves and their land.
"""

print(text.strip())
translations = {}

## Example 4

In [ ]:
text_translated = translator(text, src_lang="eng_Latn", tgt_lang="arz_Arab", max_new_tokens=256, num_beams=1, do_sample=False)
translations["arz_Arab"] = text_translated[0]["translation_text"]
print(text_translated)

## Example 5

In [ ]:
text_translated = translator(text, src_lang="eng_Latn", tgt_lang="apc_Arab", max_new_tokens=256, num_beams=1, do_sample=False)
translations["apc_Arab"] = text_translated[0]["translation_text"]
print(text_translated)

## Example 6

In [ ]:
text_translated = translator(text, src_lang="eng_Latn", tgt_lang="arb_Arab", max_new_tokens=256, num_beams=1, do_sample=False)
translations["arb_Arab"] = text_translated[0]["translation_text"]
print(text_translated)

## Example 7

In [ ]:
text_translated = translator(text, src_lang="eng_Latn", tgt_lang="ars_Arab", max_new_tokens=256, num_beams=1, do_sample=False)
translations["ars_Arab"] = text_translated[0]["translation_text"]
print(text_translated)

## Example 8

In [ ]:
text_translated = translator(text, src_lang="eng_Latn", tgt_lang="ary_Arab", max_new_tokens=256, num_beams=1, do_sample=False)
translations["ary_Arab"] = text_translated[0]["translation_text"]
print(text_translated)

## Independent regression checks

In [ ]:
import re
expected_languages = ["arz_Arab", "apc_Arab", "arb_Arab", "ars_Arab", "ary_Arab"]
assert list(translations) == expected_languages and not model.training
encoded = tokenizer(text, return_tensors="pt")
assert encoded["input_ids"].shape[1] < 512
for language, translated in translations.items():
    target_id = tokenizer.convert_tokens_to_ids(language)
    assert target_id != tokenizer.unk_token_id
    assert translated.strip() and re.search(r"[؀-ۿ]", translated)
    with torch.inference_mode():
        tokens = model.generate(**encoded, forced_bos_token_id=target_id, max_new_tokens=256, num_beams=1, do_sample=False)
    assert int(tokens[0, 1]) == target_id and int(tokens[0, -1]) == tokenizer.eos_token_id
    assert tokenizer.decode(tokens[0], skip_special_tokens=True) == translated
    print(language, ": direct generate matches pipeline; target language and EOS verified;", len(tokens[0]) - 1, "generated tokens.")
assert all(p.grad is None for p in model.parameters())
print("All five real translations, language-token routing, complete EOS termination and independent direct-generation checks passed.")

## Original source appendix

Full original blocks from raw source SHA-256 32b0a41467c88a4db0679bf632d775efc641063917b138197f57791165fa2bb2. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
!pip install transformers 
    !pip install torch
```

### Original block 2

```text
from transformers import pipeline 
import torch
```

### Original block 3

```text
translator = pipeline(task="translation",
                      model="./models/facebook/nllb-200-distilled-600M",
                      torch_dtype=torch.bfloat16)
```

### Original block 4

```text
text = """
Palestinians right to defend themselves is a fundamental aspect \
of their struggle for freedom and dignity. For decades, \
Palestinians have faced various forms of oppression, \
including occupation, displacement, and violence. \
In the face of such challenges, it is natural and justifiable for \
them to seek to protect themselves and their land.
"""
```

### Original block 5

```text
text_translated = translator(text,
                             src_lang="eng_Latn",
                             tgt_lang="arz_Arab")
text_translated
```

### Original block 6

```text
text_translated = translator(text,
                             src_lang="eng_Latn",
                             tgt_lang="apc_Arab")
```

### Original block 7

```text
text_translated = translator(text,
                             src_lang="eng_Latn",
                             tgt_lang="arb_Arab")
text_translated
```

### Original block 8

```text
text_translated = translator(text,
                             src_lang="eng_Latn",
                             tgt_lang="ars_Arab")
text_translated
```

### Original block 9

```text
text_translated = translator(text,
                             src_lang="eng_Latn",
                             tgt_lang="ary_Arab")
text_translated
```

## Original source outputs

[{‘translation_text’: ‘حق الفلسطينيين في الدفاع عن أنفسهم هو جانب أساسي من نضالهم من أجل الحرية والكرامة. لعدة عقود، واجه الفلسطينيون أشكال مختلفة من الظلم، بما في ذلك الاحتلال والنزوح والعنف. في مواجهة مثل هذه التحديات، فمن الطبيعي والمبرر لهم أن يسعون لحماية أنفسهم وأرضهم. ‘}]

[{‘translation_text’: ‘حق الفلسطينيين في الدفاع عن أنفسهم هو جانب أساسي من كفاحهم من أجل الحرية والكرامة. لعدة عقود، واجه الفلسطينيين أشكال مختلفة من الضغوط، بما في ذلك الاحتلال والنزوح والعنف. في مواجهة مثل هذه التحديات، من الطبيعي ومبرر لهم السعي لحماية أنفسهم وأرضهم. ‘}]

[{‘translation_text’: ‘حق الفلسطينيين في الدفاع عن أنفسهم هو جانب أساسي من كفاحهم من أجل الحرية والكرامة. لعدة عقود، واجه الفلسطينيين أشكال مختلفة من الضغوط، بما في ذلك الاحتلال والنزوح والعنف. في مواجهة مثل هذه التحديات، من الطبيعي ومبرر لهم السعي لحماية أنفسهم وأرضهم. ‘}]

[{‘translation_text’: ‘الحق الفلسطينيين في الدفاع عن أنفسهم هو جانب أساسي من نضالهم من أجل الحرية والكرامة. لعدة عقود، واجه الفلسطينيون أشكال مختلفة من القمع، بما في ذلك الاحتلال والنزوح والعنف. في مواجهة مثل هذه التحديات، فمن الطبيعي والمبرر لهم أن يسعون لحماية أنفسهم وأرضهم. ‘}]

[{‘translation_text’: ‘اللّهْ كَيْعْطِي الْحَقّْ لْلْفَلِسْطِينِيِّينْ بَاشْ يْدِفْعُو رْوَاحْهُمْ، وْهُوَ جْهَادْ الْحَاجَة دْيَالْهُمْ لْحُرِّيَّة وْالْكْرَامَة. وْمْنْ عَشْرِينْ عَاشْرِينْ، تّْعَدّْمُو فْالْمُوشْكَة، وْتّْهَرّْبُو، وْالْعْدَابْ. وْعْلَى وْجْهْ هَادْ التَّح’}]